# Exploration des structures BSV

Objectif : contrôler la qualité des tables `bsv_bulletins.csv` et `bsv_observations.csv`, produire des statistiques exploratoires, puis identifier les bulletins avec ou sans bioagresseurs d'intérêt pour la tomate.

Analyse détaillée des observations tomate — bioagresseurs, chronologie, météo associée, niveau de risque, indicateurs quantitatifs, tableau de synthèse exportable.


## 0. Préparation

Le notebook est conçu pour être lancé depuis le dossier `notebooks/` ou depuis la racine du projet.

In [ ]:
from pathlib import Path
import re
import unicodedata

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BULLETINS_PATH = PROJECT_ROOT / "data" / "processed" / "bsv_bulletins.csv"
OBSERVATIONS_PATH = PROJECT_ROOT / "data" / "processed" / "bsv_observations.csv"
REFERENTIEL_CNN_PATH = PROJECT_ROOT / "data" / "raw" / "referentiel_cnn_bsv.csv"

def read_csv_auto(path: Path, **kwargs) -> pd.DataFrame:
    """Read CSV files produced or supplied with mixed encodings/separators."""
    last_error = None
    for encoding in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding, sep=None, engine="python", **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def show_bar(series: pd.Series, title: str, xlabel: str = ""):
    ax = series.plot(kind="bar", figsize=(10, 4))
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Nombre")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 1. Chargement des données

In [ ]:
bulletins = read_csv_auto(BULLETINS_PATH)
observations = read_csv_auto(OBSERVATIONS_PATH)

bulletins["date_bulletin"] = pd.to_datetime(bulletins["date_bulletin"], errors="coerce")

observations_enrichies = observations.merge(
    bulletins[["bulletin_id", "region", "date_bulletin", "periode_prevision_meteo"]],
    on="bulletin_id",
    how="left",
)

print(f"Bulletins : {len(bulletins):,}")
print(f"Observations : {len(observations):,}")
display(bulletins.head())
display(observations_enrichies.head())

## 2. Contrôle qualité

In [ ]:
expected_bulletin_cols = [
    "bulletin_id", "region", "date_bulletin", "periode_prevision_meteo", "prevision_meteo"
]
expected_observation_cols = [
    "bulletin_id", "registre", "culture", "bioagresseur", "type", "departements",
    "localisations", "observation", "indicateur_quantitatif", "evolution",
    "analyse_risque", "niveau_risque_observation", "facteurs_favorables", "source_information"
]

quality = pd.DataFrame([
    {
        "table": "bulletins",
        "lignes": len(bulletins),
        "colonnes_manquantes": sorted(set(expected_bulletin_cols) - set(bulletins.columns)),
        "bulletin_id_dupliques": int(bulletins["bulletin_id"].duplicated().sum()),
        "bulletin_id_vides": int(bulletins["bulletin_id"].isna().sum()),
    },
    {
        "table": "observations",
        "lignes": len(observations),
        "colonnes_manquantes": sorted(set(expected_observation_cols) - set(observations.columns)),
        "bulletin_id_dupliques": "n/a",
        "bulletin_id_vides": int(observations["bulletin_id"].isna().sum()),
    },
])

missing_core = observations[["bulletin_id", "culture", "bioagresseur", "observation"]].isna().sum().to_frame("valeurs_vides")
observations_sans_bulletin = observations_enrichies[observations_enrichies["region"].isna()]

display(quality)
display(missing_core)
print(f"Observations sans bulletin correspondant : {len(observations_sans_bulletin):,}")
display(observations_sans_bulletin.head())

## 3. Nombre de bulletins par région

In [ ]:
bulletins_par_region = bulletins["region"].value_counts().sort_index()
display(bulletins_par_region.to_frame("nb_bulletins"))
show_bar(bulletins_par_region, "Nombre de bulletins par région", "Région")

## 4. Nombre d'observations par région

In [ ]:
observations_par_region = observations_enrichies["region"].value_counts(dropna=False).sort_index()
display(observations_par_region.to_frame("nb_observations"))
show_bar(observations_par_region, "Nombre d'observations par région", "Région")

## 5. Bioagresseurs les plus fréquents

In [ ]:
top_bioagresseurs = observations["bioagresseur"].fillna("(vide)").value_counts().head(25)
display(top_bioagresseurs.to_frame("nb_observations"))
show_bar(top_bioagresseurs.head(15), "Top 15 des bioagresseurs", "Bioagresseur")

## 6. Cultures les plus fréquentes

In [ ]:
top_cultures = observations["culture"].fillna("(vide)").value_counts().head(25)
display(top_cultures.to_frame("nb_observations"))
show_bar(top_cultures.head(15), "Top 15 des cultures", "Culture")

## 7. Observations concernant tomate general

In [ ]:
obs_norm = observations_enrichies.assign(
    culture_norm=observations_enrichies["culture"].map(normalize_text),
    bioagresseur_norm=observations_enrichies["bioagresseur"].map(normalize_text),
    observation_norm=observations_enrichies["observation"].map(normalize_text),
)

tomate_mask = (
    obs_norm["culture_norm"].str.contains(r"\btomates?\b", regex=True, na=False)
    | obs_norm["observation_norm"].str.contains(r"\btomates?\b", regex=True, na=False)
)
obs_tomate = obs_norm[tomate_mask].copy()

print(f"Observations concernant tomate : {len(obs_tomate):,}")
display(obs_tomate[[
    "bulletin_id", "region", "date_bulletin", "culture", "bioagresseur", "type",
    "departements", "observation", "source_information"
]].head(50))

## 8. Bulletins avec au moins une observation tomate

In [ ]:
bulletins_avec_tomate = (
    bulletins[bulletins["bulletin_id"].isin(obs_tomate["bulletin_id"].unique())]
    .sort_values(["region", "date_bulletin"])
)

print(f"Bulletins avec au moins une observation tomate : {len(bulletins_avec_tomate):,}")
display(bulletins_avec_tomate)

## 9. Bulletins sans observation tomate

In [ ]:
bulletins_sans_tomate = (
    bulletins[~bulletins["bulletin_id"].isin(obs_tomate["bulletin_id"].unique())]
    .sort_values(["region", "date_bulletin"])
)

print(f"Bulletins sans observation tomate : {len(bulletins_sans_tomate):,}")
display(bulletins_sans_tomate)

## 10. Profil saisonnier de la tomate

À partir de quelle semaine la tomate apparaît-elle dans les bulletins ?
On sépare les ravageurs des maladies pour caler le seuil « hors saison ».

In [ ]:
# Profil saisonnier : observations tomate par bulletin et par type
saisonnier = (
    obs_tomate[obs_tomate["region"] == "Pays de la Loire"]
    .groupby(["bulletin_id", "date_bulletin", "type"])
    .size()
    .reset_index(name="nb")
    .sort_values("date_bulletin")
)

display(saisonnier)

# Visualisation : nombre d'observations tomate par bulletin
pivot = (
    obs_tomate[obs_tomate["region"] == "Pays de la Loire"]
    .assign(mois=lambda df: df["date_bulletin"].dt.to_period("M"))
    .groupby(["mois", "type"])
    .size()
    .unstack(fill_value=0)
)

pivot.plot(kind="bar", figsize=(12, 4), title="Observations tomate par mois (Pays de la Loire)")
plt.xlabel("Mois")
plt.ylabel("Nombre d'observations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Fenêtre saisonnière par année (documentation de la variabilité)
fenetre_par_an = (
    obs_tomate[obs_tomate["region"] == "Pays de la Loire"]
    .dropna(subset=["date_bulletin"])
    .assign(annee=lambda df: df["date_bulletin"].dt.year)
    .groupby("annee")["date_bulletin"]
    .agg(debut="min", fin="max")
)
fenetre_par_an["semaine_debut"] = fenetre_par_an["debut"].dt.isocalendar().week
fenetre_par_an["semaine_fin"] = fenetre_par_an["fin"].dt.isocalendar().week
display(fenetre_par_an)

print("Fenêtre retenue dans le moteur (poolée 2023–2025) :")
print(f"  Semaine {fenetre_par_an['semaine_debut'].min()} → {fenetre_par_an['semaine_fin'].max()}")
print()
print("Conclusion : tout bulletin publié hors de cette fenêtre produira")
print("'Indéterminé / hors saison' et non 'Non corroboré' dans le moteur de compatibilité.")


In [ ]:
# Visualisation : observations tomate par mois — Grand Est
pivot_ge = (
    obs_tomate[obs_tomate["region"] == "Grand Est"]
    .assign(mois=lambda df: df["date_bulletin"].dt.to_period("M"))
    .groupby(["mois", "type"])
    .size()
    .unstack(fill_value=0)
)

pivot_ge.plot(kind="bar", figsize=(12, 4), title="Observations tomate par mois (Grand Est)")
plt.xlabel("Mois")
plt.ylabel("Nombre d'observations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Fenêtre saisonnière par année — Grand Est
fenetre_par_an_ge = (
    obs_tomate[obs_tomate["region"] == "Grand Est"]
    .dropna(subset=["date_bulletin"])
    .assign(annee=lambda df: df["date_bulletin"].dt.year)
    .groupby("annee")["date_bulletin"]
    .agg(debut="min", fin="max")
)
fenetre_par_an_ge["semaine_debut"] = fenetre_par_an_ge["debut"].dt.isocalendar().week
fenetre_par_an_ge["semaine_fin"] = fenetre_par_an_ge["fin"].dt.isocalendar().week
display(fenetre_par_an_ge)

print("Fenêtre retenue dans le moteur (poolée Grand Est) :")
print(f"  Semaine {fenetre_par_an_ge['semaine_debut'].min()} → {fenetre_par_an_ge['semaine_fin'].max()}")
print()
print("Conclusion : tout bulletin Grand Est publié hors de cette fenêtre produira")
print("'Indéterminé / hors saison' et non 'Non corroboré' dans le moteur de compatibilité.")


## 11. Bulletins avec bioagresseurs liés aux 12 classes CNN

In [ ]:
referentiel_cnn = read_csv_auto(REFERENTIEL_CNN_PATH)
display(referentiel_cnn)

def split_terms(value) -> list[str]:
    if pd.isna(value):
        return []
    return [term.strip() for term in re.split(r"[;|,]", str(value)) if term.strip()]

cnn_terms = []
for _, row in referentiel_cnn.iterrows():
    terms = []
    for col in ["nom_fr", "agent", "synonymes_bsv"]:
        if col in referentiel_cnn.columns:
            terms.extend(split_terms(row.get(col)))
    for term in terms:
        term_norm = normalize_text(term)
        if term_norm:
            cnn_terms.append({
                "classe_dataset": row.get("classe_dataset", ""),
                "terme": term,
                "terme_norm": term_norm,
            })

cnn_terms_df = pd.DataFrame(cnn_terms).drop_duplicates()
display(cnn_terms_df.sort_values(["classe_dataset", "terme"]))

matches = []
for obs_idx, obs in obs_norm.iterrows():
    haystack = f"{obs['bioagresseur_norm']} {obs['observation_norm']}"
    for _, term in cnn_terms_df.iterrows():
        pattern = rf"\b{re.escape(term['terme_norm'])}\b"
        if re.search(pattern, haystack):
            matches.append({
                "bulletin_id": obs["bulletin_id"],
                "region": obs["region"],
                "date_bulletin": obs["date_bulletin"],
                "culture": obs["culture"],
                "bioagresseur": obs["bioagresseur"],
                "classe_dataset": term["classe_dataset"],
                "terme_match": term["terme"],
                "observation": obs["observation"],
                "source_information": obs["source_information"],
            })

cnn_matches = pd.DataFrame(matches).drop_duplicates()
bulletins_cnn = bulletins[
    bulletins["bulletin_id"].isin(cnn_matches["bulletin_id"].unique())
].sort_values(["region", "date_bulletin"])

print(f"Observations liées aux classes CNN : {len(cnn_matches):,}")
print(f"Bulletins avec au moins une classe CNN : {len(bulletins_cnn):,}")
display(cnn_matches.sort_values(["classe_dataset", "region", "date_bulletin"]))
display(cnn_matches["classe_dataset"].value_counts().to_frame("nb_observations"))
display(bulletins_cnn)

## 12. Classes CNN absentes du corpus BSV

Certaines classes CNN ne produiront jamais de signal dans ce corpus, non parce que le BSV les contredit, mais parce qu'elles ne sont pas couvertes par le réseau d'observation. C'est une limite à documenter, pas un désaccord.

In [ ]:
# Classes CNN présentes vs absentes dans bsv_observations
if "cnn_matches" in dir() and len(cnn_matches) > 0:
    classes_presentes = set(cnn_matches["classe_dataset"].unique())
else:
    classes_presentes = set()

if "referentiel_cnn" in dir():
    toutes_classes = set(referentiel_cnn["classe_dataset"].dropna().unique())
else:
    toutes_classes = {
        "late_blight", "early_blight", "septoria_leaf_spot", "leaf_mold",
        "target_spot", "verticillium_wilt", "bacterial_spot", "mosaic_virus",
        "yellow_leaf_curl_virus", "spider_mites_two_spotted_spider_mite",
        "leaf_blight", "healthy"
    }

classes_absentes = toutes_classes - classes_presentes - {"healthy"}

print("Classes CNN avec au moins un signal BSV :")
for c in sorted(classes_presentes):
    print(f"  ✓ {c}")

print()
print("Classes CNN sans signal dans ce corpus (→ 'Indéterminé', pas 'Désaccord') :")
for c in sorted(classes_absentes):
    print(f"  ✗ {c}")

print()
print("Note : 'healthy' est hors logique de compatibilité.")
print("Note : 'leaf_blight' (agent indéterminé) est inclus dans le matching sur symptôme")
print("       via les synonymes 'brûlure foliaire' et 'nécrose foliaire'.")


## 13. Bioagresseurs observés sur tomate

Quels bioagresseurs sont signalés sur tomate, avec quelle fréquence, et comment se répartissent-ils entre ravageurs et maladies ?
Cette section est le point d'entrée pour calibrer le référentiel CNN–BSV sur la culture pilote.


In [ ]:
# Lookup sur les observations du corpus uniquement pour tomate
matches_tomate = [m for m in matches if m["culture"] == "tomate"]
cnn_matches_tomate = pd.DataFrame(matches_tomate).drop_duplicates()

bulletins_cnn_tomate = bulletins[
    bulletins["bulletin_id"].isin(cnn_matches_tomate["bulletin_id"].unique())
].sort_values(["region", "date_bulletin"])

print(f"Observations tomate liées aux classes CNN : {len(cnn_matches_tomate):,}")
print(f"Bulletins avec au moins une classe CNN (tomate) : {len(bulletins_cnn_tomate):,}")
display(cnn_matches_tomate.sort_values(["classe_dataset", "region", "date_bulletin"]))
display(cnn_matches_tomate["classe_dataset"].value_counts().to_frame("nb_observations"))

# Fréquence des bioagresseurs sur tomate
bio_tomate = (
    obs_tomate.groupby(["bioagresseur", "type"])
    .agg(
        nb_observations=("bulletin_id", "count"),
        nb_bulletins=("bulletin_id", "nunique"),
    )
    .reset_index()
    .sort_values("nb_observations", ascending=False)
)
display(bio_tomate.head(30))

ax = bio_tomate.set_index("bioagresseur")["nb_observations"].head(20).sort_values().plot(
    kind="barh", figsize=(10, 6)
)
ax.set_title("Top 20 bioagresseurs sur tomate")
ax.set_xlabel("Nombre d'observations")
plt.tight_layout()
plt.show()


## 14. Chronologie des signalements tomate

Pour chaque bioagresseur d'intérêt, à quelle date apparaît-il dans les bulletins ?
On visualise la présence de chaque bioagresseur semaine par semaine.


In [ ]:
# Semaine ISO de chaque observation tomate
obs_tomate_dated = obs_tomate.dropna(subset=["date_bulletin"]).copy()
obs_tomate_dated["semaine"] = obs_tomate_dated["date_bulletin"].dt.isocalendar().week.astype(int)
obs_tomate_dated["annee"] = obs_tomate_dated["date_bulletin"].dt.year
obs_tomate_dated["annee_semaine"] = (
    obs_tomate_dated["annee"].astype(str) + "-S"
    + obs_tomate_dated["semaine"].astype(str).str.zfill(2)
)

# Chronologie : nb d'observations par semaine et par bioagresseur
chrono = (
    obs_tomate_dated.groupby(["annee_semaine", "region", "bioagresseur"])
    .size()
    .reset_index(name="nb")
    .sort_values("annee_semaine")
)
display(chrono.head(40))

# Heatmap par région
for region in ["Pays de la Loire", "Grand Est"]:
    df_region = obs_tomate_dated[obs_tomate_dated["region"] == region]

    if df_region.empty:
        print(f"Aucune observation tomate pour {region}")
        continue

    top_bio_region = df_region["bioagresseur"].value_counts().head(12).index
    pivot_region = (
        df_region[df_region["bioagresseur"].isin(top_bio_region)]
        .groupby(["annee_semaine", "bioagresseur"])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.imshow(pivot_region.T.values, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(pivot_region.index)))
    ax.set_xticklabels(pivot_region.index, rotation=90, fontsize=7)
    ax.set_yticks(range(len(pivot_region.columns)))
    ax.set_yticklabels(pivot_region.columns, fontsize=8)
    ax.set_title(f"Présence des bioagresseurs sur tomate — {region} (semaine × bioagresseur)")
    plt.colorbar(im, ax=ax, label="Nb observations")
    plt.tight_layout()
    plt.show()


## 15. Météo associée aux bulletins avec observations tomate

Les bulletins contenant des observations tomate ont-ils une météo exploitable ?
On croise le statut d'extraction météo (`meteo_extraction_status`) et les prévisions textuelles disponibles.


In [ ]:
# Statut météo pour les bulletins avec tomate
meteo_cols = [
    "bulletin_id", "region", "date_bulletin",
    "nature_meteo", "meteo_extraction_status",
    "meteo_extraction_method", "meteo_extraction_score",
    "periode_prevision_meteo", "prevision_meteo",
]
meteo_tomate = bulletins_avec_tomate[
    [c for c in meteo_cols if c in bulletins_avec_tomate.columns]
].copy()

print("Statut météo dans les bulletins avec observations tomate :")
display(meteo_tomate["meteo_extraction_status"].value_counts().to_frame("nb"))

print("\nRépartition région × statut météo :")
display(
    meteo_tomate.groupby(["region", "meteo_extraction_status"]).size()
    .unstack(fill_value=0)
)

print("\nExemples de prévisions textuelles lisibles :")
display(
    meteo_tomate[meteo_tomate["meteo_extraction_status"] == "lisible"]
    [["bulletin_id", "region", "date_bulletin", "prevision_meteo"]]
    .dropna(subset=["prevision_meteo"])
    .head(15)
)


## 16. Niveau de risque et analyse de risque sur tomate

Les observations tomate contiennent-elles des informations sur le niveau de risque ou des analyses de risque ? Ces champs alimentent directement le moteur de compatibilité.


In [ ]:
# Niveau de risque renseigné
print("Niveau de risque renseigné :")
display(obs_tomate["niveau_risque_observation"].replace("", float("nan")).value_counts(dropna=False).to_frame("nb"))

print("\nAnalyse de risque renseignée :")
display(obs_tomate["analyse_risque"].replace("", float("nan")).value_counts(dropna=False).head(20).to_frame("nb"))

print("\nFacteurs favorables renseignés :")
display(obs_tomate["facteurs_favorables"].replace("", float("nan")).value_counts(dropna=False).head(20).to_frame("nb"))

# Observations tomate avec au moins un champ risque renseigné
obs_tomate_risque = obs_tomate[
    obs_tomate["niveau_risque_observation"].fillna("").ne("")
    | obs_tomate["analyse_risque"].fillna("").ne("")
    | obs_tomate["facteurs_favorables"].fillna("").ne("")
].copy()

print(f"\nObservations tomate avec au moins un champ risque : {len(obs_tomate_risque):,}")
display(
    obs_tomate_risque[[
        "bulletin_id", "date_bulletin", "bioagresseur",
        "niveau_risque_observation", "analyse_risque", "facteurs_favorables", "observation"
    ]].head(20)
)

total = len(obs_tomate)
avec_risque = obs_tomate["niveau_risque_observation"].fillna("").ne("").sum()
print(f"Observations tomate avec niveau de risque : {avec_risque} / {total} ({100*avec_risque/total:.1f}%)")
print(f"Observations tomate sans niveau de risque : {total - avec_risque} / {total} ({100*(total-avec_risque)/total:.1f}%)")

print("\nTaux de renseignement du niveau de risque par template :")
display(
    obs_tomate.merge(
        bulletins[["bulletin_id", "template_id"]], on="bulletin_id", how="left"
    ).groupby("template_id")["niveau_risque_observation"]
    .apply(lambda s: f"{s.fillna('').ne('').sum()} / {len(s)} ({s.fillna('').ne('').mean():.0%})")
    .to_frame("taux_renseigne")
)

## 17. Indicateurs quantitatifs sur tomate

Quelles observations contiennent des pourcentages ou des comptages d'individus ?
Ce sont les observations les plus exploitables pour qualifier un niveau de pression.


In [ ]:
# Observations avec indicateur quantitatif non vide
obs_tomate_quant = obs_tomate[
    obs_tomate["indicateur_quantitatif"].notna()
    & obs_tomate["indicateur_quantitatif"].ne("")
].copy()

print(f"Observations tomate avec indicateur quantitatif : {len(obs_tomate_quant):,} / {len(obs_tomate):,}")
display(
    obs_tomate_quant[[
        "bulletin_id", "date_bulletin", "bioagresseur",
        "indicateur_quantitatif", "observation"
    ]].head(30)
)

# Bioagresseurs les mieux documentés quantitativement
quant_par_bio = (
    obs_tomate_quant.groupby("bioagresseur")
    .agg(
        nb_obs_quant=("bulletin_id", "count"),
        pct_total=(
            "bulletin_id",
            lambda s: round(100 * len(s) / max(1, obs_tomate["bioagresseur"].eq(s.name).sum()), 1)
        ),
    )
    .reset_index()
    .sort_values("nb_obs_quant", ascending=False)
)
display(quant_par_bio)


## 18. Tableau de synthèse par bulletin × bioagresseur (tomate)

Table pivot finale : pour chaque bulletin contenant une observation tomate, on rassemble date, région, bioagresseur, type, indicateur quantitatif et météo disponible.
C'est la vue qui servira de base pour tester le moteur de compatibilité.


In [ ]:
# Jointure observations tomate × bulletins (météo incluse)
synthese_tomate = obs_tomate_dated.merge(
    bulletins[[
        "bulletin_id", "meteo_extraction_status", "prevision_meteo",
        "periode_prevision_meteo", "nature_meteo"
    ]].rename(columns={"periode_prevision_meteo": "periode_prevision_meteo_bulletin"}),
    on="bulletin_id",
    how="left",
).assign(
    meteo_disponible=lambda df: df["meteo_extraction_status"].eq("lisible")
)[[
    "bulletin_id", "region", "date_bulletin", "annee", "semaine",
    "culture", "bioagresseur", "type",
    "departements", "indicateur_quantitatif",
    "niveau_risque_observation", "analyse_risque", "facteurs_favorables",
    "observation",
    "meteo_disponible", "periode_prevision_meteo_bulletin", "prevision_meteo",
]].sort_values(["date_bulletin", "bioagresseur"])

print(f"Lignes dans la synthèse tomate : {len(synthese_tomate):,}")
display(synthese_tomate)

#Export
synthese_tomate.to_csv(
     PROJECT_ROOT / "data" / "processed" / "synthese_tomate.csv",
     index=False, encoding="utf-8-sig" )


## 19. Synthèse rapide

Cette cellule rassemble les indicateurs principaux pour faciliter le reporting.

In [ ]:
synthese = pd.DataFrame([
    {"indicateur": "bulletins_total",                "valeur": len(bulletins)},
    {"indicateur": "observations_total",             "valeur": len(observations)},
    {"indicateur": "observations_tomate",            "valeur": len(obs_tomate)},
    {"indicateur": "bulletins_avec_tomate",          "valeur": len(bulletins_avec_tomate)},
    {"indicateur": "bulletins_sans_tomate",          "valeur": len(bulletins_sans_tomate)},
    {"indicateur": "bulletins_avec_classes_cnn",     "valeur": len(bulletins_cnn)},
    {"indicateur": "fenetre_saisonniere_pdl_debut",  "valeur": int(fenetre_par_an["semaine_debut"].min())},
    {"indicateur": "fenetre_saisonniere_pdl_fin",    "valeur": int(fenetre_par_an["semaine_fin"].max())},
    {"indicateur": "fenetre_saisonniere_ge_debut",   "valeur": int(fenetre_par_an_ge["semaine_debut"].min())},
    {"indicateur": "fenetre_saisonniere_ge_fin",     "valeur": int(fenetre_par_an_ge["semaine_fin"].max())},
    {"indicateur": "classes_cnn_avec_signal_bsv",    "valeur": len(classes_presentes)},
    {"indicateur": "classes_cnn_sans_signal_bsv",    "valeur": len(classes_absentes)},
    {"indicateur": "bioagresseurs_distincts_tomate", "valeur": obs_tomate["bioagresseur"].nunique()},
    {"indicateur": "obs_tomate_avec_quant",          "valeur": len(obs_tomate_quant)},
    {"indicateur": "obs_tomate_avec_risque",         "valeur": len(obs_tomate_risque)},
    {"indicateur": "bulletins_tomate_meteo_lisible", "valeur": (meteo_tomate["meteo_extraction_status"] == "lisible").sum()},
])
display(synthese)